# Multi-Hop Wikipedia QA — Colab / Kaggle / Binder companion

This notebook mirrors the local `uv` project from the course's
[Build a Multi-Hop Question-Answering Tool Over a Small Wikipedia Sample](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/multihop-wikipedia-qa)
lesson, adapted to run in a hosted notebook with no local files: a small corpus of
Wikipedia-style articles, local embeddings with `sentence-transformers`, NumPy
cosine-similarity retrieval, and a free-tier LLM for the final answers.

See the [lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/multihop-wikipedia-qa) for the full walkthrough and the
[local example project](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/multihop-wikipedia-qa) for the real, file-based version of this same code.

## Why two hops?

A plain RAG app retrieves the top chunks for a question **once** and asks the model
to answer from them. That works when the answer lives in one document — but a whole
class of questions need facts from *two* documents combined, and a single retrieval
pass usually finds only one of them. The model then has to guess the rest, and you
get a plausible-sounding but **wrong** answer.

This notebook runs the same question through **two pipelines side by side**:

- **single-hop**: retrieve the top-K chunks once, answer from only those.
- **multi-hop**: retrieve, ask the model whether the evidence is enough, and if not
  retrieve a **second round** guided by the model's own follow-up search query, then
  answer from the merged evidence.

Both print the evidence chunks they used, so you can audit exactly what happened.


In [ ]:
!pip install -q sentence-transformers numpy openai python-dotenv

## The sample corpus

Same six articles as the local example's `data/articles/` — short, plain-text,
Wikipedia-style summaries (fictional but realistic: biographies, companies, cities,
an event). They're *crafted* so that a few questions genuinely need facts from two
articles at once. A hosted notebook has no local files, so they're embedded here
directly as strings.


In [ ]:
ARTICLES = {
    "amina-rahman.md": "# Amina Rahman\n\n**Amina Rahman** (born 1975) is a Bangladeshi biochemist known for developing the industrial extraction process used to make protein from microalgae.\n\n## Early life and education\n\nRahman was born in 1975 in Dhaka, the capital of Bangladesh. She studied chemistry at the University of Dhaka and earned a PhD in biochemistry from the same institution in 2001, writing her dissertation on protein recovery from single-celled organisms.\n\n## Career\n\nAfter a postdoctoral fellowship in Singapore, Rahman returned to the University of Dhaka as a lecturer. In 2011 she was recruited by a Swiss biotechnology startup as its chief scientist, where she led the development of a scalable method for extracting protein from cultivated microalgae.\n\nHer extraction method, which uses a mild enzymatic treatment instead of harsh solvents, became the production process behind the company's flagship product. Industry analysts credited the method's low cost with making algae-based protein commercially viable for the first time.\n\n## Recognition\n\nIn 2018 Rahman received the international Women in Biotechnology Award for her work on sustainable protein production. She has published more than forty peer-reviewed papers and holds three patents related to microalgae processing.\n\n## Personal life\n\nRahman continues to live and work in Switzerland, but travels regularly to Bangladesh, where she mentors students at her alma mater and funds a small scholarship for women in the natural sciences.",
    "basel.md": "# Basel\n\n**Basel** is the third-largest city in Switzerland, located on the Rhine River where the Swiss, French, and German borders meet. It is a major center of the pharmaceutical and life-sciences industries.\n\n## Geography and population\n\nBasel sits at the point where Switzerland, France, and Germany come together, giving it the nickname \"the three-country city\". Its metropolitan population is just over 500,000, and the city is a major railway and freight hub for central Europe.\n\n## Life sciences industry\n\nBasel is home to some of the world's largest pharmaceutical companies, along with a dense cluster of smaller biotechnology startups. One such startup, the microalgae protein company Cereolabs, was founded in this city in 2009 by a group of researchers who had left a nearby university institute. The city's research hospitals and university labs are a major draw for young scientists.\n\n## The Basel Climate Accord\n\nIn 2019, Basel hosted the signing of the **Basel Climate Accord**, an international agreement in which participating governments committed to cutting greenhouse-gas emissions from road transport. The accord's signing ceremony was held at the city's conference center, and the agreement is named after the city as a result.\n\nSeveral transport and battery-industry firms later cited the accord as a driver of demand for electric vehicles, and analysts frequently referenced it when discussing the growth of the electric bus market in Europe.\n\n## Tourism and culture\n\nBasel is known for its art museums, including a prominent collection of modern and contemporary art, and for its annual carnival, one of the largest in Europe. The city's old town is a popular destination for weekend visitors from the neighboring countries.",
    "cereolabs.md": "# Cereolabs\n\n**Cereolabs** is a Swiss biotechnology company that produces protein from cultivated microalgae. It is best known for launching **AquaPro**, the first commercially available algae-based protein powder for human consumption.\n\n## History\n\nThe company was founded in 2009 by a group of former university researchers, and built its first pilot facility two years later. It remained a small research firm for its early years, funding itself through government grants and a single early angel investment.\n\n## Products\n\nCereolabs' flagship product, AquaPro, launched in 2017. It is a neutral-tasting protein powder made from microalgae grown in closed bioreactors, marketed as a sustainable alternative to soy and whey protein. The production process is built around a mild enzymatic extraction method developed by the company's chief scientist, who joined the firm in 2011.\n\nThe company also sells a concentrated algae paste, marketed under the name \"Aqualift\", to food manufacturers as an ingredient for plant-based meat products.\n\n## Facilities\n\nCereolabs operates its headquarters and main production facility in an industrial district on the outskirts of its home city, along with a second laboratory opened in Lisbon in 2022. The Lisbon lab focuses on consumer product formulation.\n\n## Business\n\nIn 2021, following strong sales of AquaPro in European supermarkets, Cereolabs raised a Series B funding round led by a London-based investment firm. The company has stated that it plans to open a production plant in North America by 2028.",
    "elena-marchetti.md": "# Elena Marchetti\n\n**Elena Marchetti** (born 1963) is an Italian electrical engineer best known for founding one of Europe's earliest dedicated lithium-ion battery manufacturers.\n\n## Early life and education\n\nMarchetti was born in 1963 in the city of Naples, in southern Italy. She studied electrical engineering at the Polytechnic University of Turin, where she focused on electrochemical energy storage and wrote a thesis on the thermal management of rechargeable battery packs.\n\n## Career\n\nIn 1992, at the age of 29, Marchetti founded a battery manufacturing company in the city of Turin and served as its first chief executive. The company grew slowly for a decade, surviving on small contracts from electric forklift makers and uninterruptible-power-supply vendors.\n\nHer breakthrough came in 2009, when her battery-pack design won the European Energy Innovation Prize, a continental award recognizing engineering achievements in clean energy. The win raised her company's profile significantly and led to its first major public-transit contracts.\n\n## Later life\n\nMarchetti stepped down as chief executive in 2019 and retired in 2020, moving to Lisbon, Portugal, where she advises early-stage energy startups and writes occasional essays on battery recycling. She has no children and prefers to keep a low public profile.\n\n## Legacy\n\nIndustry historians credit Marchetti with popularizing the practice of pairing battery chemistry research directly with vehicle-integration engineering — an approach that was unusual for a small firm in the 1990s. Her original company continues to operate today under a different name.",
    "lisbon.md": "# Lisbon\n\n**Lisbon** is the capital and largest city of Portugal, located on the Atlantic coast of the Iberian Peninsula. It has been an important trading port since the Age of Discovery and remains the country's economic and cultural center.\n\n## Geography and population\n\nLisbon sits on seven hills at the mouth of the Tagus River, facing the Atlantic Ocean. Its metropolitan area has a population of roughly 2.9 million, about a quarter of Portugal's total population. The city's mild, sunny climate makes it a popular destination for remote workers and retirees.\n\n## Public transport\n\nThe city's public transport system is operated by **TransLisboa**, the municipal transit authority, which runs the city's trams, buses, and metro. TransLisboa is known for its historic yellow tram network, which climbs the city's steepest streets.\n\nIn 2016, TransLisboa launched Europe's first fully electric bus route, a single line connecting the city center to the airport, using battery packs supplied by an Italian battery manufacturer. The route's success led TransLisboa to expand electric buses to several more lines over the following years.\n\n## Culture and events\n\nLisbon hosts the annual Lisbon Tech Summit, a three-day technology conference that draws startups and investors from across Europe. The event is held each October at the city's riverside convention center.\n\n## Notable residents\n\nThe city has a growing community of tech entrepreneurs and retired engineers. Among its well-known residents is a celebrated Italian electrical engineer who moved to Lisbon after retiring, and who advises early-stage energy startups from her home in the Alfama district.",
    "volta-dynamics.md": "# Volta Dynamics\n\n**Volta Dynamics** was an Italian manufacturer of lithium-ion battery packs for electric commercial vehicles, headquartered in Turin. It was founded in the early 1990s by the winner of the 2009 European Energy Innovation Prize, and operated under this name until 2021.\n\n## History\n\nThe company was founded in the early 1990s in Turin, Italy, growing out of a university research project on rechargeable battery systems. For its first fifteen years it supplied relatively small battery packs to electric forklift and delivery-vehicle makers across northern Italy.\n\nThe firm's first significant public-transit win came in 2015, when it won a contract to supply battery systems for the bus fleet of Lisbon's public transport operator. The following year, in 2016, that operator launched Europe's first fully electric bus route, powered by Volta Dynamics battery packs.\n\n## Products\n\nVolta Dynamics focused exclusively on stationary and vehicle-mounted battery packs, deliberately avoiding consumer electronics. Its core product line was the \"TransPack\" series of swappable battery modules, designed so that a depot could charge a fresh pack while a bus continued its route on another.\n\n## Renaming and later years\n\nIn 2019 the company went public on the Milan Stock Exchange. In 2021 it merged with a German EV-grid startup and renamed itself Voltora. Under that name, the business shifted its focus toward grid-scale energy storage and battery reuse for renewable-power utilities.\n\n## Recognition\n\nBattery industry trade publications repeatedly cited Volta Dynamics in their annual rankings of European battery integrators, praising the reliability record of its TransPack modules in daily transit use."
}

print(f"Loaded {len(ARTICLES)} articles: {list(ARTICLES.keys())}")

## Split into chunks, embed locally

Same chunking logic as `main.py`: split on blank lines into paragraphs, then greedily
re-merge short paragraphs up to a target size. Then embed every chunk with
`all-MiniLM-L6-v2` — fully local, no API key, no cost — keeping the vectors in memory
since a notebook has no `data/index.npy` to persist to.


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

TARGET_CHUNK_SIZE = 600  # characters


def split_into_paragraphs(text: str) -> list[str]:
    paragraphs = [p.strip() for p in text.split("\n\n")]
    return [p for p in paragraphs if p]


def merge_short_paragraphs(paragraphs: list[str], target_size: int) -> list[str]:
    chunks = []
    current = ""
    for paragraph in paragraphs:
        if current and len(current) + len(paragraph) > target_size:
            chunks.append(current)
            current = paragraph
        else:
            current = f"{current}\n\n{paragraph}" if current else paragraph
    if current:
        chunks.append(current)
    return chunks


def chunk_articles(articles: dict[str, str]) -> list[dict]:
    chunks = []
    for source, text in articles.items():
        paragraphs = split_into_paragraphs(text)
        for chunk_text in merge_short_paragraphs(paragraphs, TARGET_CHUNK_SIZE):
            chunks.append({"text": chunk_text, "source": source})
    return chunks


chunks = chunk_articles(ARTICLES)
print(f"Chunked {len(ARTICLES)} articles into {len(chunks)} chunks")

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode([c["text"] for c in chunks], normalize_embeddings=True)
print(f"Embedded {embeddings.shape[0]} chunks ({embeddings.shape[1]}-dim)")

In [ ]:
def retrieve(question: str, top_k: int = 3) -> list[dict]:
    """Returns the top_k chunks most similar to `question`, each with its
    cosine-similarity score, ranked highest first."""
    q = model.encode([question], normalize_embeddings=True)[0]
    similarities = embeddings @ q
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [
        {**chunks[i], "score": float(similarities[i])}
        for i in top_indices
    ]


# Quick demo on a single-article question:
for r in retrieve("What is the name of Lisbon's public transit authority?"):
    print(f"{r['score']:.3f}  [{r['source']}]  {' '.join(r['text'].split())[:70]}...")

## Get a free-tier LLM API key

Answer *generation* (the last step of both pipelines) needs a free-tier LLM API —
retrieval itself is fully local and needs no key. Pick any provider from the table in
the [lesson's Setup section](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/multihop-wikipedia-qa#get-a-free-llm-api-key);
**GitHub Models** is the suggested default since it needs no separate signup (a
personal access token with the `models: read` scope from
[github.com/settings/tokens](https://github.com/settings/tokens)).

The key is entered with `getpass` so it never gets typed into a visible cell or saved
into this notebook's output — never hardcode a real API key here.


In [ ]:
import os
from getpass import getpass

LLM_PROVIDER = "github"  # github (default) | gemini | groq | mistral | cerebras | openrouter

ENV_VAR_BY_PROVIDER = {
    "github": "GITHUB_TOKEN",
    "gemini": "GOOGLE_API_KEY",
    "groq": "GROQ_API_KEY",
    "mistral": "MISTRAL_API_KEY",
    "cerebras": "CEREBRAS_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
}

env_var = ENV_VAR_BY_PROVIDER[LLM_PROVIDER]
os.environ[env_var] = getpass(f"Enter your {LLM_PROVIDER} API key ({env_var}): ")

## The two pipelines

Same prompts and provider setup as `main.py`. The `SUFFICIENCY_PROMPT` is the heart
of the multi-hop path: it asks the model to either answer (`SUFFICIENT`) or to
declare the evidence incomplete and write a follow-up search query that would find the
missing fact (`INSUFFICIENT`). `multi_hop` then retrieves a second round with that
query and answers from the merged evidence.


In [ ]:
from openai import OpenAI

ANSWER_PROMPT = """Answer the question using ONLY the context below. If the
context doesn't contain the answer, say so plainly -- do not make something up.

Context:
{context}

Question: {question}

Answer:"""

SUFFICIENCY_PROMPT = """You get ONE retrieval pass of evidence, which may not
be enough to answer the question -- the answer might need facts that live in a
document this retrieval didn't return.

Context:
{context}

Question: {question}

Decide whether the Context above contains enough information to answer the
Question. Reply with exactly one of these two forms:

If YES -- SUFFICIENT, followed by your answer on the next line(s):
    SUFFICIENT
    <answer using only the context>

If NO -- do NOT try to answer. Reply:
    INSUFFICIENT
    <one follow-up search query, single line, that would find the missing
    information -- name the specific entity or fact you need>

Never output both forms."""


def _build_client():
    bases = {
        "github": ("https://models.github.ai/inference", "GITHUB_TOKEN"),
        "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY"),
        "groq": ("https://api.groq.com/openai/v1", "GROQ_API_KEY"),
        "mistral": ("https://api.mistral.ai/v1", "MISTRAL_API_KEY"),
        "cerebras": ("https://api.cerebras.ai/v1", "CEREBRAS_API_KEY"),
        "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY"),
    }
    base_url, env_var = bases[LLM_PROVIDER]
    return OpenAI(api_key=os.environ[env_var], base_url=base_url)


MODELS = {
    "github": "gpt-4o-mini",
    "gemini": "gemini-3.5-flash",
    "groq": "llama-3.3-70b-versatile",
    "mistral": "mistral-small-latest",
    "cerebras": "llama-3.3-70b",
    "openrouter": "meta-llama/llama-3.3-70b-instruct:free",
}


def chat(messages: list[dict]) -> str:
    response = _build_client().chat.completions.create(
        model=MODELS[LLM_PROVIDER],
        messages=messages,
        temperature=0.2,
    )
    return response.choices[0].message.content


def format_context(chunks: list[dict]) -> str:
    return "\n\n".join(f"[{c['source']}] {c['text']}" for c in chunks)


def parse_sufficiency(text: str) -> tuple[str, str]:
    """Returns ("sufficient", answer) or ("insufficient", followup_query)."""
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    if lines and "INSUFFICIENT" in lines[0].upper():
        followup = lines[1] if len(lines) > 1 else ""
        return "insufficient", followup
    answer = "\n".join(lines[1:]) if lines and lines[0].upper().startswith("SUFFICIENT") else text
    return "sufficient", answer.strip()


def merge_dedupe(*chunk_lists: list[dict]) -> list[dict]:
    seen: set[str] = set()
    merged: list[dict] = []
    for chunk_list in chunk_lists:
        for chunk in chunk_list:
            if chunk["text"] not in seen:
                seen.add(chunk["text"])
                merged.append(chunk)
    return merged


def single_hop(question: str, top_k: int = 3) -> tuple[str, list[dict]]:
    """Baseline: retrieve once, answer from that one retrieval pass."""
    retrieved = retrieve(question, top_k=top_k)
    prompt = ANSWER_PROMPT.format(context=format_context(retrieved), question=question)
    answer = chat([{"role": "user", "content": prompt}])
    return answer, retrieved


def multi_hop(question: str, top_k: int = 3) -> tuple[str, list[dict], str | None, list[list[dict]]]:
    """Two rounds: retrieve, check sufficiency, and if needed retrieve again
    guided by the model's follow-up query, then answer from merged evidence."""
    round1 = retrieve(question, top_k=top_k)
    verdict = chat([{"role": "user", "content": SUFFICIENCY_PROMPT.format(
        context=format_context(round1), question=question)}])
    status, followup = parse_sufficiency(verdict)
    if status == "sufficient":
        return followup, round1, None, [round1]
    round2 = retrieve(followup, top_k=top_k)
    combined = merge_dedupe(round1, round2)
    final_prompt = ANSWER_PROMPT.format(context=format_context(combined), question=question)
    answer = chat([{"role": "user", "content": final_prompt}])
    return answer, combined, followup, [round1, round2]

## Run the comparison on a genuinely multi-hop question

Here's the money shot: a question whose answer only exists once two articles are
combined. Run this cell and watch what happens — single-hop gets the *clue* article
but not the *fact* article, while multi-hop spots the gap, writes a follow-up query,
pulls the second article, and answers correctly.

Run it a few times (or on `--query`-style questions of your own) — on such a small
corpus single-hop will sometimes *accidentally* land the right chunks and get a
multi-hop question right by luck. That's a real property of small corpora, not a bug;
the evidence printed beside each answer is the honest audit trail.


In [ ]:
def show(label: str, answer: str, evidence: list[dict]) -> None:
    print(f"--- {label} ---")
    print(f"Answer: {answer.strip()}")
    print("Evidence used:")
    for i, c in enumerate(evidence, 1):
        snippet = " ".join(c["text"].split())
        print(f"  {i}. [{c['source']}] score {c['score']:.3f}: {snippet[:110]}...")
    print()


question = "Who founded the company that powered TransLisboa's electric buses?"

single_answer, single_evidence = single_hop(question)
multi_answer, multi_evidence, followup, rounds = multi_hop(question)

show("SINGLE-HOP", single_answer, single_evidence)

extra = ""
if followup:
    extra = f"Round 2 (guided by follow-up query): {followup.strip()}"
else:
    extra = "Evidence was sufficient in round 1 -- no second retrieval needed."
show(f"MULTI-HOP ({extra})", multi_answer, multi_evidence)

Built your own version of this? See the course's
[`examples/student-projects/`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/student-projects)
gallery to share it with the class. Welcome to writing Python outside the browser. 🎓
